In [1]:
# analysis
import numpy as np
import pandas as pd 

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# single-cell
import scanpy as sc
import liana as li
import mudata as mu
import omnipath as op

from pathlib import Path
import os 

# seting global dir
cwd=Path.cwd()
if cwd.parent.name == "notebooks":
    os.chdir(cwd.parent.parent) 
print(os.getcwd())

# constants
PATH_DATA = Path('data/multimodal')
MOUSE_TAX_ID = 10090


/home/maxi7524/micromamba/envs/OmniPhysiBoss_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


/home/maxi7524/repositories/project_computational_biology_initial_conditions_pymyboss


In [6]:
import anndata
import mudata
import scanpy as sc
import liana as li
import pandas as pd
from pathlib import Path
from typing import Dict, Any, Tuple

# Functional Graph and Bivariate Pipe Execution Layer
def run_liana_multimodal_pipeline(
    mdata: mudata.MuData,
    modality_key: str = "rna",
    resource_name: str = "mouseconsensus",
    connectivity_key: str = "spatial_connectivities",
    liana_key: str = "liana_res"
) -> mudata.MuData:
    """
    Executes an isolated functional pipeline to calculate spatial neighbor graphs 
    and bivariate signaling scores inside a MuData container structure.

    :param mdata: Multimodal data container holding the omics spaces.
    :type mdata: mudata.MuData
    :param modality_key: Target modality token representing expression data, defaults to "rna".
    :type modality_key: str
    :param resource_name: Name of the ligand-receptor database, defaults to "mouseconsensus".
    :type resource_name: str
    :param connectivity_key: Target key for storage of the adjacency matrix, defaults to "spatial_connectivities".
    :type connectivity_key: str
    :param liana_key: Target key under .uns where results are stored, defaults to "liana_res".
    :type liana_key: str
    :return: Processed MuData instance with populated spatial annotations.
    :rtype: mudata.MuData
    """
    # Parameter derivation and optimization phase
    ## Extract coordinate matrix and resolve target spatial modality
    adata_mod = mdata[modality_key]
    spatial_coords = adata_mod.obsm['spatial']
    base_conn_key = connectivity_key.replace("_connectivities", "")

    ## Dynamic bandwidth scanning loop (safeguarded query boundary)
    ### We scale the scan boundary up to 50_000 to ensure connectivity constraints
    print(f"\n[-] Scanning spatial density distributions for modality: '{modality_key}'...")
    _, df_bandwidth = li.ut.query_bandwidth(
        coordinates=spatial_coords,
        start=0,
        end=50_000,
        interval_n=30
    )

    ### Extract valid bandwidth constants satisfying the strict k >= 6 condition
    valid_rows = df_bandwidth[df_bandwidth['neighbours'] >= 6]
    if not valid_rows.empty:
        min_bandwidth = float(valid_rows['bandwith'].min())
        print(f" -> Optimal mathematical bandwidth isolated (k >= 6): {min_bandwidth}")
    else:
        min_bandwidth = float(df_bandwidth['bandwith'].max())
        print(f"[!] Warning: Safe bandwidth bound (k >= 6) unreached. Utilizing max fallback: {min_bandwidth}")

    # Execution of spatial graph assembly
    ## Package optimized neighbor configuration constraints
    neighbors_config = {
        'adata': adata_mod,
        "kernel": "gaussian",
        "bandwidth": min_bandwidth,
        "cutoff": 0.1,
        "set_diag": False,
        "key_added": base_conn_key
    }

    print(f"[-] Computing spatial neighbors into modality '{modality_key}'.obsp['{connectivity_key}']...")
    li.ut.spatial_neighbors(**neighbors_config)

    # Execution of local bivariate scoring
    ## Package multimodal configuration matrices with explicit mod filters
    bivariate_config = {
        'mdata': mdata,
        'x_mod': modality_key,
        'y_mod': modality_key,
        'local_name': 'cosine',
        'global_name': None,
        'resource_name': resource_name,
        'n_perms': None,
        'mask_negatives': True,
        'seed': 1337,
        'connectivity_key': connectivity_key
    }

    print(f"[-] Computing bivariate ligand-receptor cross-correlation scores...")
    bivariate_res = li.mt.bivariate(**bivariate_config)

    ## Map data structures back into core modality workspaces
    if bivariate_res is not None:
        mdata[modality_key].uns[liana_key] = bivariate_res
        print(f"[✓] Local bivariate metrics mapped to mdata['{modality_key}'].uns['{liana_key}']")
    else:
        print("[!] Critical: Bivariate execution returned empty context framework.")

    return mdata

In [13]:

# ładujemy dane 
# create ann data objects
rna = sc.read(PATH_DATA / "sma_rna.h5ad") # 
msi = sc.read(PATH_DATA / "sma_msi.h5ad")
ct = sc.read(PATH_DATA / "sma_deconv.h5ad")

# create multimodal data
mdata = mu.MuData({'rna':rna, 'msi':msi, 'ct':ct})
print(mdata)

# ??? 
# important mouse data
## resources for liana
mouse_resource = li.rs.select_resource(resource_name='mouseconsensus')

run_liana_multimodal_pipeline(mdata, 'rna', 'mouseconsensus')

/home/maxi7524/micromamba/envs/OmniPhysiBoss_env/lib/python3.12/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
/home/maxi7524/micromamba/envs/OmniPhysiBoss_env/lib/python3.12/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.


MuData object with n_obs × n_vars = 6041 × 17782
  3 modalities
    rna:	3036 × 16486
      obs:	'in_tissue', 'array_row', 'array_col', 'x', 'y', 'lesion', 'region', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts'
      var:	'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
      uns:	'lesion_colors', 'log1p', 'region_colors', 'spatial'
      obsm:	'spatial'
      layers:	'counts'
    msi:	3005 × 1248
      obs:	'x', 'y', 'array_row', 'array_col', 'leiden', 'n_counts', 'index_right', 'region', 'lesion'
      var:	'mean', 'std', 'mz', 'max_intensity', 'mz_raw', 'annotated'
      uns:	'leiden', 'leiden_colors', 'log1p', 'neig

/home/maxi7524/micromamba/envs/OmniPhysiBoss_env/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.


ValueError: `shape` is inconsistent with `obs` (6041 rows instead of 3036)

In [12]:
import anndata
import mudata
import scanpy as sc
import liana as li
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from pathlib import Path
from typing import Dict, Any, Tuple

def run_liana_multimodal_pipeline(
    mdata: mudata.MuData,
    modality_key: str = "rna",
    resource_name: str = "mouseconsensus",
    connectivity_key: str = "spatial_connectivities",
    liana_key: str = "liana_res"
) -> mudata.MuData:
    """
    Executes a functional pipeline to calculate spatial neighbor graphs 
    and bivariate signaling scores natively inside a MuData container structure.
    """
    # Parameter derivation and optimization phase
    ## Extract coordinate matrix and resolve target spatial modality
    adata_mod = mdata[modality_key]
    spatial_coords = adata_mod.obsm['spatial']
    base_conn_key = connectivity_key.replace("_connectivities", "")

    ## Dynamic bandwidth scanning loop (safeguarded query boundary)
    print(f"\n[-] Scanning spatial density distributions for modality: '{modality_key}'...")
    _, df_bandwidth = li.ut.query_bandwidth(
        coordinates=spatial_coords,
        start=0,
        end=50_000,
        interval_n=30
    )

    ### Extract valid bandwidth constants satisfying the strict k >= 6 condition
    valid_rows = df_bandwidth[df_bandwidth['neighbours'] >= 6]
    if not valid_rows.empty:
        min_bandwidth = float(valid_rows['bandwith'].min())
        print(f" -> Optimal mathematical bandwidth isolated (k >= 6): {min_bandwidth}")
    else:
        min_bandwidth = float(df_bandwidth['bandwith'].max())
        print(f"[!] Warning: Safe bandwidth bound (k >= 6) unreached. Utilizing max fallback: {min_bandwidth}")

    # Execution of spatial graph assembly
    ## Package optimized neighbor configuration constraints
    neighbors_config = {
        'adata': adata_mod,
        "kernel": "gaussian",
        "bandwidth": min_bandwidth,
        "cutoff": 0.1,
        "set_diag": False,
        "key_added": base_conn_key
    }

    print(f"[-] Computing spatial neighbors into modality '{modality_key}'.obsp['{connectivity_key}']...")
    li.ut.spatial_neighbors(**neighbors_config)

    # CRITICAL MULTIMODAL FIX: Construct a global matrix matching mdata.n_obs shape
    ## We build a zero matrix of size (n_obs, n_obs) and inject the sub-modality graph
    print(f"[-] Compiling dimensional-aligned connectivity matrix for global MuData space...")
    n_global = mdata.n_obs
    global_adj = csr_matrix((n_global, n_global), dtype=np.float32).tolil()
    
    ## Identify integer position mapping from sub-modality names to global indices
    global_obs_names = mdata.obs_names.tolist()
    global_name_to_idx = {name: idx for idx, name in enumerate(global_obs_names)}
    mod_indices = [global_name_to_idx[name] for name in adata_mod.obs_names]
    
    ## Squeeze sub-modality CSR matrix blocks into the allocated global template
    local_adj = adata_mod.obsp[connectivity_key].tocoo()
    for i, j, val in zip(local_adj.row, local_adj.col, local_adj.data):
        global_adj[mod_indices[i], mod_indices[j]] = val
        
    mdata.obsp[connectivity_key] = global_adj.tocsr()

    # Execution of local bivariate scoring
    ## We explicitly force 'x_use_raw'=False and 'y_use_raw'=False to bypass md.raw exceptions
    bivariate_config = {
        'mdata': mdata,
        'x_mod': modality_key,
        'y_mod': modality_key,
        'local_name': 'cosine',
        'global_name': None,
        'resource_name': resource_name,
        'n_perms': None,
        'mask_negatives': True,
        'seed': 1337,
        'connectivity_key': connectivity_key,
        'x_use_raw': False,
        'y_use_raw': False
    }

    print(f"[-] Computing bivariate ligand-receptor cross-correlation scores...")
    bivariate_res = li.mt.bivariate(**bivariate_config)

    ## Map data structures back into core modality workspaces
    if bivariate_res is not None:
        mdata[modality_key].uns[liana_key] = bivariate_res
        print(f"[✓] Local bivariate metrics mapped to mdata['{modality_key}'].uns['{liana_key}']")
    else:
        print("[!] Critical: Bivariate execution returned empty context framework.")

    return mdata